import libraries

In [ ]:
import os
import json
import pandas as pd
import requests
from dotenv import load_dotenv 

load api key from .env 

In [ ]:
load_dotenv()


In [ ]:

api_key = os.getenv("API_KEY")

create get request for teams rankings

In [ ]:
url = "https://v1.formula-1.api-sports.io/rankings/teams"

season = 2024

headers = {
  'x-apisports-key': api_key,
}
query_parameters = {'season': season }

response_teams = requests.get(url, headers=headers, params=query_parameters)


print(response_teams.status_code)

In [ ]:
payload_teams = response_teams.json()

In [ ]:
payload_teams

create get request for driver rankings

In [ ]:
url = "https://v1.formula-1.api-sports.io/rankings/drivers"

season = 2024

headers = {
  'x-apisports-key': api_key,
}

query_parameters = {'season': season }

response_drivers = requests.get(url, headers=headers, params=query_parameters)

print(response_drivers.status_code)

In [ ]:
payload_drivers = response_drivers.json()

In [ ]:
payload_drivers

# ====EXTRACTION====

In [ ]:
print(json.dumps(payload_drivers, indent=5))


In [ ]:
payload_teams['response'][0]['team']['logo']

Save all images

In [ ]:
import urllib.request

In [ ]:

"""for t in payload_teams['response']:
    url = t['team']['logo']
    team_id = t['team']['id']
    filename = f"{team_id}.png"
    urllib.request.urlretrieve(url, filename)

print("Image saved successfully!")"""


In [ ]:
"""for d in payload_drivers['response']:
    url = d['driver']['image']
    driver_id = d['driver']['id']
    filename = f"{driver_id}.png"
    urllib.request.urlretrieve(url, filename)

print("Image saved successfully!")"""

In [ ]:
payload_teams['response'][0]['team']['logo']

create paths for images

In [ ]:
img_folder_drivers = r"../assets/images/drivers"
img_folder_teams = r"../assets/images/teams"

In [ ]:

img_paths_drivers = [
    os.path.join(img_folder_drivers, file)
    for file in os.listdir(img_folder_drivers)
    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))

]

img_paths_teams = [
    os.path.join(img_folder_teams, file)
    for file in os.listdir(img_folder_teams)
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))
]

In [ ]:
import os
import re

def numeric_sort_key(path):
    filename = os.path.basename(path)
    stem = os.path.splitext(filename)[0]
    match = re.match(r"^(\d+)", stem)
    if match:
        return int(match.group(1))
    return float("inf")

img_paths_drivers = sorted(img_paths_drivers, key=numeric_sort_key)
img_paths_teams = sorted(img_paths_teams, key=numeric_sort_key)

print(img_paths_drivers)
print(img_paths_teams)

In [ ]:
for t in img_paths_teams:
    print(t)

In [ ]:
team_rows = []
team_col_names = [ 
    'team_id',
    'team_name', 
    'team_position',
    'team_points',
    'season',
    'team_logo'
]

In [ ]:
payload_teams['response'][0]

In [ ]:

sorted_teams = payload_teams['response'] = sorted(
    payload_teams['response'],
    key=lambda team: team['team']['id']
)

In [ ]:
sorted_teams

In [ ]:
team_rows.clear()

In [ ]:
img_index = 0

for team in sorted_teams:
    t_id = team['team']['id']
    t_name = team['team']['name']
    t_position = team['position']
    t_points = team['points']
    t_season = team['season']
    t_logo = img_paths_teams[img_index]
    img_index+=1
    team_rows.append((t_id, t_name, t_position, t_points, t_season, t_logo,))
    

In [ ]:

for t in team_rows:
    print(t)

In [ ]:
driver_rows = []
driver_col_names = [
    'driver_id',
    'driver_position',
    'driver_name',
    'abbreviation',
    'driver_wins', 
    'driver_points',
    'season',
    'driver_image',
    'team_id'
]

In [ ]:
driver_rows.clear()

In [ ]:
payload_drivers['response']

In [ ]:
sorted_drivers = payload_drivers['response'] = sorted(
    payload_drivers['response'],
    key=lambda d: d['driver']['id']
)

In [ ]:
driver_rows.clear()

img_index = 0
for d in sorted_drivers:
    d_id = d['driver']['id']
    d_position = d['position']
    d_name = d['driver']['name']
    d_abbr = d['driver']['abbr'] or "None"
    d_wins = d['wins']
    d_points = d['points'] or 0
    d_season = d['season']
    d_team_id = d['team']['id'] or 0

    d_image = img_paths_drivers[img_index]

    driver_rows.append(
        (d_id, d_position, d_name, d_abbr, d_wins, d_points, d_season, d_image, d_team_id)
    )

print(f"driver_rows count: {len(driver_rows)}")

for row in driver_rows:
    print(row)

# creating dataframe

In [ ]:
df_teams = pd.DataFrame(team_rows, columns=team_col_names)
df_teams

In [ ]:
df_drivers = pd.DataFrame(driver_rows, columns=driver_col_names)
# update driver's missing team id
df_drivers.loc[df_drivers["driver_id"] == 89, "team_id"] = 7
# update missing abbreviation
df_drivers.loc[df_drivers["driver_id"] == 92, "abbreviation"] = "SAR"
df_drivers.loc[df_drivers["driver_id"] == 83, "abbreviation"] = "ZHO"
df_drivers.loc[df_drivers["driver_id"] == 105, "abbreviation"] = "COL"





In [ ]:
df_drivers

# ===LOADING===

In [ ]:
load_dotenv()

In [ ]:
sql_host        = os.getenv("HOST")
sql_port        = os.getenv("MYSQL_PORT")
sql_user        = os.getenv("MYSQL_USER")
sql_password    = os.getenv("MYSQL_PASSWORD")
sql_database    = os.getenv("MYSQL_DATABASE")

print(f"{sql_host}, {sql_port}, {sql_user}, {sql_password}, {sql_database}")

In [ ]:
from mysql import connector

In [ ]:
server_connection = connector.connect(
    host=sql_host,
    port=sql_port,
    user=sql_user,
    password=sql_password,
    connection_timeout = 10,
    autocommit=False,
    raise_on_warnings=True
)

server_cursor = server_connection.cursor()
print("Connected Successfully")

server_cursor.close()
server_connection.close()

print("closed successfully")

connect to database

In [ ]:
db_connection = connector.connect(
    host=sql_host,
        port=sql_port,
        user=sql_user,
        password=sql_password,
        database=sql_database
        
)

db_cursor = db_connection.cursor()
print("Connected successfully")


start upsert operation

In [ ]:
# get list of tuples from each data frame - teams and drivers
teams_tuples_list = list(df_teams.itertuples(index=False, name= None))

drivers_tuples_list = list(df_drivers.itertuples(index=False, name= None))


In [ ]:
print(teams_tuples_list)
print(drivers_tuples_list)

In [ ]:
table_teams = "teams"
table_drivers = "drivers"


In [ ]:
print(team_col_names)

In [ ]:
query_add_teams = f""" 

    INSERT INTO {table_teams}
    (team_id, team_name, team_position, team_points, season, team_logo)
    VALUES 
    (%s, %s, %s, %s, %s, %s) AS src
    
    ON DUPLICATE KEY UPDATE
        
    team_name = src.team_name,
    team_position = src.team_position,
    team_points = src.team_points,
    team_logo = src.team_logo

"""

In [ ]:
no_rows_uploaded = len(teams_tuples_list)

In [ ]:
try:
    db_cursor.executemany(query_add_teams, teams_tuples_list)
    db_connection.commit()
    print(f"{no_rows_uploaded} rows added")
except Exception as e:
    db_connection.rollback()
    print(e)    

In [ ]:
query_add_drivers = f"""

    INSERT INTO {table_drivers}
    (driver_id, driver_position, driver_name, abbreviation, driver_wins, driver_points, season, driver_image, team_id)
    VALUES
    (%s, %s, %s, %s, %s, %s, %s, %s, %s) AS src

    ON DUPLICATE KEY UPDATE

    driver_position = src.driver_position,
    driver_name = src.driver_name,
    abbreviation = src.abbreviation,
    driver_wins = src.driver_wins,
    driver_points = src.driver_points,
    season = src.season,
    driver_image = src.driver_image,
    team_id = src.team_id

"""

In [ ]:
# execute many insert/upsert for drivers
no_rows_uploaded = len(drivers_tuples_list)

try:
    db_cursor.executemany(query_add_drivers, drivers_tuples_list)
    db_connection.commit()
    print(f"{no_rows_uploaded} driver rows added/updated")
except Exception as e:
    db_connection.rollback()
    print(f"Error inserting drivers: {e}")

In [ ]:
db_cursor.close()
db_connection.close()

# == Saving data ==

In [117]:

df_teams.to_csv('teams.csv', index = 'False')
df_drivers.to_csv('drivers.csv', index = 'False')